# Notebook 47 — From Lindblad Dynamics to the Rate Distribution

This version is Colab-friendly. It installs missing dependencies automatically before imports.

The goal is to connect the real simulated system to the synthetic mechanism derived earlier:

physical parameters  
→ open-system Lindblad dynamics  
→ effective-rate distribution  
→ stretching exponent `b`


In [ ]:
# Colab / notebook dependency bootstrap
import importlib
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pip_name])

for import_name, pip_name in [
    ("numpy", None),
    ("matplotlib", None),
    ("scipy", None),
    ("qutip", None),
]:
    ensure_package(import_name, pip_name)

print("Dependencies ready.")


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar, curve_fit
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Baseline protocol and nearby family

In [ ]:
baseline = {
    "T": 26.0,
    "alpha": 0.10,
    "omega_max": 1.09,
    "delta0": 1.00,
    "p": 1.00,
    "q": 0.72,
}
V = 4.0

T0 = baseline["T"]
alpha0 = baseline["alpha"]
omega0 = baseline["omega_max"]

protocols = {
    "baseline": dict(baseline),
    "T_minus": {**baseline, "T": 24.0},
    "T_plus": {**baseline, "T": 30.0},
    "alpha_minus": {**baseline, "alpha": 0.08},
    "alpha_plus": {**baseline, "alpha": 0.12},
    "omega_minus": {**baseline, "omega_max": 1.02},
    "omega_plus": {**baseline, "omega_max": 1.16},
}


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)

    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)

    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    H = build_time_dependent_hamiltonian(
        params["T"], params["omega_max"], params["alpha"], V, params["delta0"], params["p"], params["q"]
    )
    times = np.linspace(0.0, params["T"], n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            vals.append(basis_state.overlap(state))
        else:
            val = basis_state.dag() * state * basis_state
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, "full") else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j * b), np.exp(1j * a), np.exp(1j * (a + b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            score = np.abs(basis_state.overlap(final_state)) ** 2
        else:
            val = basis_state.dag() * final_state * basis_state
            score = np.real(val.full()[0, 0]) if hasattr(val, "full") else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Noise grid

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 13)
gamma_phi_vals = np.linspace(0.0, 0.12, 13)


## Evaluate one protocol

In [ ]:
def evaluate_protocol(params):
    cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            U_eff, finals = build_noisy_effective_map(
                params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=140
            )
            cz_map[i, j] = compensated_cz_fidelity(U_eff)
            coh_map[i, j] = coherence_proxy(finals)
            leak_map[i, j] = leakage_from_finals(finals)

    S = cz_map * coh_map * (1.0 - leak_map)
    S_norm = S / np.max(S) if np.max(S) > 0 else S.copy()

    S_gamma = S_norm[0, :]
    S_phi = S_norm[:, 0]
    interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind="linear", fill_value="extrapolate")

    def loss(lam):
        mapped = lam * gamma_phi_vals
        pred = interp_gamma(np.clip(mapped, gamma_decay_vals.min(), gamma_decay_vals.max()))
        return float(np.mean((pred - S_phi) ** 2))

    fit = minimize_scalar(loss, bounds=(0.1, 5.0), method="bounded")
    lam = float(fit.x)

    gamma_eff_grid = np.zeros_like(S_norm)
    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            gamma_eff_grid[i, j] = gamma_decay + lam * gamma_phi

    return {
        "params": params,
        "S_norm": S_norm,
        "lambda": lam,
        "gamma_eff_grid": gamma_eff_grid,
    }


## Boundary laws for T_c

In [ ]:
def build_boundary_laws():
    baseline_res = evaluate_protocol(baseline)
    lambda0 = baseline_res["lambda"]

    def score_for_params(params):
        res = evaluate_protocol(params)
        S = res["S_norm"]
        S_gamma = S[0, :]
        S_phi = S[:, 0]
        interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind="linear", fill_value="extrapolate")

        def axis_loss(lam):
            mapped = lam * gamma_phi_vals
            pred = interp_gamma(np.clip(mapped, gamma_decay_vals.min(), gamma_decay_vals.max()))
            return float(np.mean((pred - S_phi) ** 2))

        aloss = axis_loss(res["lambda"])

        bS = baseline_res["S_norm"]
        bge = baseline_res["gamma_eff_grid"].ravel()
        bsv = bS.ravel()
        order = np.argsort(bge)
        bge = bge[order]
        bsv = bsv[order]
        bbins = np.linspace(bge.min(), bge.max(), 13)
        bcenters = 0.5 * (bbins[:-1] + bbins[1:])
        bmeans = np.full(len(bcenters), np.nan)
        for k in range(len(bcenters)):
            mask = (bge >= bbins[k]) & (bge < bbins[k + 1])
            if np.any(mask):
                bmeans[k] = np.mean(bsv[mask])
        bvalid = ~np.isnan(bmeans)
        interp0 = interp1d(bcenters[bvalid], bmeans[bvalid], kind="linear", fill_value="extrapolate")

        ge = res["gamma_eff_grid"].ravel()
        sv = S.ravel()
        order = np.argsort(ge)
        ge = ge[order]
        sv = sv[order]
        bins = np.linspace(ge.min(), ge.max(), 13)
        centers = 0.5 * (bins[:-1] + bins[1:])
        means = np.full(len(centers), np.nan)
        for k in range(len(centers)):
            mask = (ge >= bins[k]) & (ge < bins[k + 1])
            if np.any(mask):
                means[k] = np.mean(sv[mask])
        valid = ~np.isnan(means)
        curve_mismatch = float(np.mean((means[valid] - interp0(np.clip(centers[valid], bcenters[bvalid].min(), bcenters[bvalid].max()))) ** 2))
        lambda_shift = abs(res["lambda"] - lambda0)

        return float(np.exp(
            -(lambda_shift / 0.15)
            -(aloss / 0.001)
            -(curve_mismatch / 0.001)
        ))

    boundary_level = 0.30
    T_vals_local = np.array([20.0, 24.0, 26.0, 30.0, 34.0])
    alpha_vals_local = np.array([0.06, 0.08, 0.10, 0.12, 0.14])
    omega_vals_local = np.array([0.95, 1.02, 1.09, 1.16, 1.23])

    TA_score = np.zeros((len(alpha_vals_local), len(T_vals_local)))
    TO_score = np.zeros((len(omega_vals_local), len(T_vals_local)))

    for i, alpha in enumerate(alpha_vals_local):
        for j, T in enumerate(T_vals_local):
            TA_score[i, j] = score_for_params({**baseline, "T": float(T), "alpha": float(alpha)})

    for i, omega in enumerate(omega_vals_local):
        for j, T in enumerate(T_vals_local):
            TO_score[i, j] = score_for_params({**baseline, "T": float(T), "omega_max": float(omega)})

    def extract_vertical_boundary(score_map, xvals, yvals, level):
        bx, by = [], []
        for i, y in enumerate(yvals):
            row = score_map[i, :]
            crossing = None
            for j in range(len(xvals) - 1):
                v0, v1 = row[j], row[j + 1]
                if (v0 >= level and v1 < level) or (v0 > level and v1 <= level):
                    x0, x1 = xvals[j], xvals[j + 1]
                    xc = x0 if v1 == v0 else x0 + (level - v0) * (x1 - x0) / (v1 - v0)
                    crossing = xc
                    break
            if crossing is None and np.max(row) >= level and np.min(row) >= level:
                crossing = float(xvals[-1])
            if crossing is not None:
                bx.append(float(crossing))
                by.append(float(y))
        return np.array(bx), np.array(by)

    TA_bx, TA_by = extract_vertical_boundary(TA_score, T_vals_local, alpha_vals_local, boundary_level)
    TO_bx, TO_by = extract_vertical_boundary(TO_score, T_vals_local, omega_vals_local, boundary_level)

    TA_x = TA_by / alpha0
    TA_y = TA_bx / T0
    TO_x = TO_by / omega0
    TO_y = TO_bx / T0

    coeff_alpha = np.polyfit(TA_x - 1.0, TA_y, deg=1)
    coeff_omega = np.polyfit(TO_x, TO_y, deg=1)
    return coeff_alpha, coeff_omega

coeff_alpha, coeff_omega = build_boundary_laws()
coeff_alpha, coeff_omega


## Helper functions for collapse and fitting

In [ ]:
def Tc_over_T0_alpha(alpha):
    x = alpha / alpha0
    return np.polyval(coeff_alpha, x - 1.0)

def Tc_over_T0_omega(omega):
    x = omega / omega0
    return np.polyval(coeff_omega, x)

def Tc_from_params(params):
    Tc_alpha = T0 * Tc_over_T0_alpha(params["alpha"])
    Tc_omega = T0 * Tc_over_T0_omega(params["omega_max"])
    return float(np.sqrt(np.maximum(Tc_alpha, 1e-9) * np.maximum(Tc_omega, 1e-9)))

def stretched_law(x, a, b):
    return np.exp(-a * np.power(x, b))

def fit_stretched(x, S):
    x = np.clip(np.asarray(x, dtype=float), 1e-12, None)
    S = np.clip(np.asarray(S, dtype=float), 1e-12, 1.0)
    popt, _ = curve_fit(
        stretched_law,
        x,
        S,
        p0=[5.0, 1.0],
        bounds=([0.0, 0.1], [100.0, 5.0]),
        maxfev=50000,
    )
    a, b = [float(v) for v in popt]
    pred = stretched_law(x, a, b)
    mse = float(np.mean((S - pred) ** 2))
    return {"a": a, "b": b, "pred": pred, "mse": mse}

def rate_stats(samples):
    samples = np.asarray(samples, dtype=float)
    mean = float(np.mean(samples))
    std = float(np.std(samples))
    cv = float(std / mean) if mean > 0 else np.nan
    q10, q50, q90 = np.quantile(samples, [0.10, 0.50, 0.90])
    width_ratio = float(q90 / q10) if q10 > 0 else np.nan
    return {
        "mean": mean,
        "std": std,
        "cv": cv,
        "q10": float(q10),
        "q50": float(q50),
        "q90": float(q90),
        "width_ratio": width_ratio,
    }


## Rebuild family and recover the optimal collapse exponent

In [ ]:
family = {}
for name, pset in protocols.items():
    family[name] = evaluate_protocol(pset)

def collapse_error_nu(nu):
    x_all = []
    s_all = []
    for name, res in family.items():
        T = res["params"]["T"]
        Tc = Tc_from_params(res["params"])
        x = res["gamma_eff_grid"].ravel() * (T / Tc) ** nu
        s = res["S_norm"].ravel()
        x_all.extend(list(x))
        s_all.extend(list(s))
    x_all = np.array(x_all, dtype=float)
    s_all = np.array(s_all, dtype=float)

    order = np.argsort(x_all)
    x_all = x_all[order]
    s_all = s_all[order]

    bins = np.linspace(x_all.min(), x_all.max(), 24)
    centers = 0.5 * (bins[:-1] + bins[1:])
    means = np.full(len(centers), np.nan)
    for k in range(len(centers)):
        mask = (x_all >= bins[k]) & (x_all < bins[k + 1])
        if np.any(mask):
            means[k] = np.mean(s_all[mask])

    valid = ~np.isnan(means)
    pred = interp1d(centers[valid], means[valid], kind="linear", fill_value="extrapolate")
    p = pred(np.clip(x_all, centers[valid].min(), centers[valid].max()))
    return float(np.mean((s_all - p) ** 2))

opt = minimize_scalar(collapse_error_nu, bounds=(0.1, 3.0), method="bounded")
nu_opt = float(opt.x)
print("Recovered nu =", nu_opt)


## Build per-protocol collapsed curves

In [ ]:
protocol_curves = {}

for name, res in family.items():
    T = res["params"]["T"]
    Tc = Tc_from_params(res["params"])
    x = res["gamma_eff_grid"].ravel() * (T / Tc) ** nu_opt
    s = res["S_norm"].ravel()

    order = np.argsort(x)
    x = np.asarray(x[order], dtype=float)
    s = np.asarray(s[order], dtype=float)

    bins = np.linspace(np.min(x), np.max(x), 28)
    centers = 0.5 * (bins[:-1] + bins[1:])
    means = np.full(len(centers), np.nan)
    for k in range(len(centers)):
        mask = (x >= bins[k]) & (x < bins[k + 1])
        if np.any(mask):
            means[k] = np.mean(s[mask])

    valid = ~np.isnan(means)
    x_curve = centers[valid]
    S_curve = means[valid]

    fit = fit_stretched(x_curve[1:], S_curve[1:])
    protocol_curves[name] = {
        "x": x_curve,
        "S": S_curve,
        "fit": fit,
        "T": T,
        "Tc": Tc,
        "T_over_Tc": T / Tc,
    }

for name, obj in protocol_curves.items():
    print(name, "b =", obj["fit"]["b"], "MSE =", obj["fit"]["mse"])


## Extract local effective-rate samples from the real system

In [ ]:
real_rate_rows = []

for name, obj in protocol_curves.items():
    x = np.asarray(obj["x"], dtype=float)
    S = np.asarray(obj["S"], dtype=float)

    mask = (x > np.quantile(x, 0.08)) & (x < np.quantile(x, 0.92)) & (S > 1e-8) & (S < 1.0)
    x_mid = x[mask]
    S_mid = S[mask]

    logS = np.log(np.clip(S_mid, 1e-12, 1.0))
    Gamma = -np.gradient(logS, x_mid)

    finite = np.isfinite(Gamma) & (Gamma > 0)
    Gamma = Gamma[finite]
    x_mid = x_mid[finite]

    stats = rate_stats(Gamma)

    real_rate_rows.append({
        "protocol": name,
        "b_real": obj["fit"]["b"],
        "a_real": obj["fit"]["a"],
        "Gamma_samples": Gamma,
        "x_samples": x_mid,
        **stats,
    })

for row in real_rate_rows:
    print(
        row["protocol"],
        "b=", f'{row["b_real"]:.3f}',
        "cv=", f'{row["cv"]:.3f}',
        "width=", f'{row["width_ratio"]:.3f}',
        "meanG=", f'{row["mean"]:.3f}',
    )


## Plot the real protocol-level collapsed curves

In [ ]:
plt.figure(figsize=(8.2, 5.3))
for row in real_rate_rows:
    obj = protocol_curves[row["protocol"]]
    plt.plot(obj["x"], obj["S"], label=f'{row["protocol"]}, b={row["b_real"]:.2f}')
plt.xlabel("x = gamma_eff * (T/Tc)^nu")
plt.ylabel("S(x)")
plt.title("Real collapsed protocol curves")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Plot extracted real rate distributions

In [ ]:
plt.figure(figsize=(8.6, 5.5))
for row in real_rate_rows:
    Gamma = row["Gamma_samples"]
    hist, edges = np.histogram(Gamma, bins=16, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    plt.plot(centers, hist, label=f'{row["protocol"]}, CV={row["cv"]:.2f}')
plt.xlabel("Gamma samples from real collapsed curves")
plt.ylabel("density")
plt.title("Extracted effective-rate distributions from Lindblad data")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Real-system relation: b vs CV

In [ ]:
cv_real = np.array([row["cv"] for row in real_rate_rows], dtype=float)
b_real = np.array([row["b_real"] for row in real_rate_rows], dtype=float)

plt.figure(figsize=(7.2, 5.0))
plt.scatter(cv_real, b_real, s=70)
for row in real_rate_rows:
    plt.annotate(row["protocol"], (row["cv"], row["b_real"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.xlabel("coefficient of variation of real Gamma samples")
plt.ylabel("real fitted stretching exponent b")
plt.title("Real-system mapping: rate heterogeneity vs b")
plt.tight_layout()
plt.show()


## Rebuild the synthetic b(CV) law from Notebook 46

In [ ]:
rng = np.random.default_rng(42)

def ensemble_decay(rates, x):
    rates = np.asarray(rates, dtype=float)
    return np.array([np.mean(np.exp(-rates * xi)) for xi in x], dtype=float)

def fit_stretched_simple(x, S):
    popt, _ = curve_fit(
        stretched_law,
        np.clip(x, 1e-12, None),
        np.clip(S, 1e-12, 1.0),
        p0=[1.0, 1.0],
        bounds=([0.0, 0.1], [100.0, 5.0]),
        maxfev=50000,
    )
    a, b = [float(v) for v in popt]
    return a, b

def make_gamma_rates(mean_rate, shape_k, n=20000):
    theta = mean_rate / shape_k
    return rng.gamma(shape=shape_k, scale=theta, size=n)

def make_lognormal_rates(mean_rate, sigma, n=20000):
    mu = np.log(mean_rate) - 0.5 * sigma**2
    return rng.lognormal(mean=mu, sigma=sigma, size=n)

x_syn = np.linspace(0.0, 0.30, 220)
synthetic_rows = []

for k in [0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0]:
    rates = make_gamma_rates(mean_rate=8.0, shape_k=k, n=20000)
    S = ensemble_decay(rates, x_syn)
    _, b = fit_stretched_simple(x_syn[1:], S[1:])
    st = rate_stats(rates)
    synthetic_rows.append({"family": "gamma", "control": k, "b": b, **st})

for sigma in [0.10, 0.20, 0.35, 0.50, 0.70, 0.90, 1.10]:
    rates = make_lognormal_rates(mean_rate=8.0, sigma=sigma, n=20000)
    S = ensemble_decay(rates, x_syn)
    _, b = fit_stretched_simple(x_syn[1:], S[1:])
    st = rate_stats(rates)
    synthetic_rows.append({"family": "lognormal", "control": sigma, "b": b, **st})

cv_syn = np.array([r["cv"] for r in synthetic_rows], dtype=float)
b_syn = np.array([r["b"] for r in synthetic_rows], dtype=float)

def b_model(cv, alpha, beta):
    return 1.0 / (1.0 + alpha * np.power(cv, beta))

popt_b, _ = curve_fit(
    b_model,
    cv_syn,
    b_syn,
    p0=[1.0, 1.0],
    bounds=([0.0, 0.0], [100.0, 10.0]),
    maxfev=50000,
)
alpha_fit, beta_fit = [float(v) for v in popt_b]
print("Synthetic b(CV) law:")
print("alpha =", alpha_fit)
print("beta  =", beta_fit)


## Overlay real-system points on the synthetic b(CV) law

In [ ]:
cv_grid = np.linspace(min(np.min(cv_syn), np.min(cv_real)) * 0.9, max(np.max(cv_syn), np.max(cv_real)) * 1.05, 400)

plt.figure(figsize=(8.0, 5.5))
plt.scatter(cv_syn, b_syn, s=45, alpha=0.45, label="synthetic cases")
plt.plot(cv_grid, b_model(cv_grid, alpha_fit, beta_fit), linewidth=2.5, label="synthetic b(CV) fit")
plt.scatter(cv_real, b_real, s=85, marker="D", label="real Lindblad-derived points")

for row in real_rate_rows:
    plt.annotate(row["protocol"], (row["cv"], row["b_real"]), fontsize=8, xytext=(4, 4), textcoords="offset points")

plt.xlabel("coefficient of variation of Gamma")
plt.ylabel("stretched exponent b")
plt.title("Do real Lindblad-derived rates follow the synthetic b(CV) law?")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Residual of real points to the synthetic law

In [ ]:
b_pred_real = b_model(cv_real, alpha_fit, beta_fit)
resid_real = b_real - b_pred_real
mse_real_to_synlaw = float(np.mean(resid_real ** 2))

plt.figure(figsize=(7.4, 4.8))
plt.axhline(0.0, linestyle="--")
plt.scatter(cv_real, resid_real, s=70)
for row, resid in zip(real_rate_rows, resid_real):
    plt.annotate(row["protocol"], (row["cv"], resid), fontsize=8, xytext=(4, 4), textcoords="offset points")
plt.xlabel("coefficient of variation of real Gamma samples")
plt.ylabel("real b − synthetic-law prediction")
plt.title("Residual of real-system points to the synthetic b(CV) law")
plt.tight_layout()
plt.show()

print("MSE of real points to synthetic b(CV) law =", mse_real_to_synlaw)


## Compact summary

In [ ]:
lines = []
lines.append("From Lindblad dynamics to the rate distribution")
lines.append("")
lines.append(f"Recovered collapse exponent nu = {nu_opt:.6f}")
lines.append(f"Synthetic b(CV) law: b ≈ 1 / (1 + {alpha_fit:.6f} * CV^{beta_fit:.6f})")
lines.append(f"Real-to-synthetic-law MSE = {mse_real_to_synlaw:.6e}")
lines.append("")
lines.append("Real-system protocol summaries:")
for row in real_rate_rows:
    lines.append(
        f'- {row["protocol"]}: b={row["b_real"]:.6f}, CV={row["cv"]:.6f}, '
        f'width={row["width_ratio"]:.6f}, meanΓ={row["mean"]:.6f}, stdΓ={row["std"]:.6f}'
    )
lines.append("")
lines.append("Interpretation:")
lines.append("- The real system generates a measurable distribution of local effective rates.")
lines.append("- Its heterogeneity can be summarized by CV.")
lines.append("- If the real points lie near the synthetic b(CV) curve, then the stretching exponent is predictable from the Lindblad-generated rate distribution.")
lines.append("- This closes the mechanism loop: physical controls -> Lindblad dynamics -> rate distribution -> stretching exponent b.")

print("\n".join(lines))


## Final conclusion

This notebook links the real simulated system to the synthetic mechanism derived earlier.

It extracts effective-rate samples directly from the Lindblad-generated collapsed curves, computes their heterogeneity, and compares the resulting `(CV, b)` points to the synthetic `b(CV)` law.

If the real points fall near the synthetic curve, then the project has a predictive bridge:

physical parameters  
→ open-system Lindblad dynamics  
→ effective-rate distribution  
→ stretching exponent `b`
